In [ ]:
import sys
import os
basedir = ""


sys.path.append(os.path.join("scripts"))
# import VectorandImageHandlers
from FileHandlers.VideoHandlers.NPVideoHandler import NPVideoHandler
from GenerateBehTrack.WormTrack_062225 import WormTrack
from GenerateBehTrack.BehaviorClassifier import BehaviorClassifier


import numpy as np
import matplotlib.pyplot as plt
import os

# SCRIPT_DIR = os.path.dirname(os.path.abspath(os.path.join("..","VectorandImageHandlers")))
# sys.path.append(os.path.dirname(SCRIPT_DIR))
# SCRIPT_DIR = os.path.dirname(os.path.abspath(os.path.join("..","FileHandlers")))
# sys.path.append(os.path.dirname(SCRIPT_DIR))


def get_start_frame_from_trackID(trackID):
    start_frame, end_frame = trackID.split("_")
    return int(start_frame)

def get_filename_from_trackID(trackID):
    start_frame, end_frame = trackID.split("_")
    return f"({start_frame}, {end_frame})"

n_frames = 14880



import ast
trackIDs_save_dir = os.path.join(basedir, "features/")
trackIDs = []
with open(os.path.join(trackIDs_save_dir, "trackIDs.txt"), "r", encoding="utf-8") as file:
    for line in file:
        trackID_str, frame_start, frame_end = ast.literal_eval(line.strip())
        trackIDs.append((trackID_str, frame_start, frame_end))
print(trackIDs)
# centroids = wormtrack.bbox_img_centroids+box_buffer_starts*px_to_um
# midline =  (wormtrack.oriented_midlines +box_buffer_starts)*px_to_um


n_tracks= len(trackIDs)

centroids_mat = np.zeros((n_frames, n_tracks, 2))*np.nan
midline_mat = np.zeros((n_frames, n_tracks,15,  2))*np.nan
mdpt_coord_mat= np.zeros((n_frames, n_tracks, 2))*np.nan
tail_coord_mat = np.zeros((n_frames, n_tracks, 2))*np.nan
head_coord_mat = np.zeros((n_frames, n_tracks, 2))*np.nan
is_looping_mat =  np.zeros((n_frames, n_tracks))*np.nan

parent_dir = os.path.join(basedir, "background_subtracted_jpegs/atr0his0_compressed")
param_yaml = os.path.join(basedir, "background_subtracted_jpegs/atr0his0_compressed")
# parent_dir = "/Users/friederikebuck/Desktop/Thesis/FERAL_scripts/Final/background_subtracted_jpegs/atr0his0_compressed"


param_yaml = os.path.join(basedir, "scripts/beh_config.yaml")

for i, (trackID_str, frame_start, frame_end) in enumerate(trackIDs):
    trackID = get_filename_from_trackID(trackID_str)
    background_subtracted_dir = os.path.join(parent_dir,trackID_str, "background_subtracted", "npy")
    vid_handler = NPVideoHandler(background_subtracted_dir, is_compressed=False)

    box_buffer_starts_file = os.path.join(basedir, "background_subtracted_jpegs","bbox_coords", f"{trackID_str}_bbox_buffer_starts.csv")
    box_buffer_starts= np.loadtxt(box_buffer_starts_file)
    start_frame = get_start_frame_from_trackID(trackID_str)
    
    wormtrack = WormTrack(param_yaml)
    wormtrack.init_by_phil_tracker_output(vid_handler, start_frame, box_buffer_starts)
    wormtrack.get_midlines()
    
    
    # wormtrack.create_and_save_debug_imgs(debug_dir_track)
    
    chunk_start_is, chunk_end_is = wormtrack.get_midline_chunk_indices()
    
    behClass = BehaviorClassifier("", param_yaml)
    
    
    behClass.init_from_WormTrack_obj(wormtrack, interpolated = False)
    speed = behClass.get_behavior_classification(save_debug_images = False)
    chunk_start_is, chunk_end_is = wormtrack.get_midline_chunk_indices()
    wormtrack = behClass.flip_head_tail_based_on_forward_rev_lengths(wormtrack, chunk_start_is, chunk_end_is, chunk_length_thres = 125)


    
    '''
    shift midline by bbox buffer coords; multiple by number 
    save midlines
    
    multiple centroids by number 
    
    
    save is looping 
    
    save good midline frames 
    
    save speed(?) 
    
    label_to_mat["centroids"],
    label_to_mat["head_coord"],   
    label_to_mat["tail_coord"],   
    label_to_mat["mdpt_coord"],
    label_to_mat["is_looping"], 
    
    
    combine into mat
    
    
    '''
    is_looping_track = wormtrack.is_looping.astype('uint8')
    px_to_um = 15.4
    centroids = behClass.global_centroids*px_to_um
    midline =  (wormtrack.oriented_midlines)*px_to_um 
    n_pts = 15
    mdpt_coord = midline[:,int(n_pts/2) ]
    tail_coord = midline[:,-1]
    head_coord = midline[:, 0]
    

    centroids_mat[start_frame:frame_end+1, i] = centroids
    midline_mat[start_frame:frame_end+1, i] = midline
    
    
    mdpt_coord_mat[start_frame:frame_end+1,i] = mdpt_coord
    tail_coord_mat[start_frame:frame_end+1, i] = tail_coord
    head_coord_mat[start_frame:frame_end+1,i] = head_coord
    is_looping_mat[start_frame:frame_end+1,i] = is_looping_track



[('0_13', 0, 4734), ('3129_7', 3129, 5164), ('3359_8', 3359, 4584), ('4786_8', 4786, 5916), ('0_2', 0, 6735), ('0_12', 0, 3342), ('0_3', 0, 6912), ('3423_10', 3423, 10511), ('0_8', 0, 2924), ('0_10', 0, 12788), ('0_11', 0, 687), ('1096_5', 1096, 2546), ('7897_8', 7897, 13660), ('865_8', 865, 5865), ('3455_5', 3455, 5017), ('0_9', 0, 687), ('6636_1', 6636, 6925), ('0_5', 0, 3308), ('2997_7', 2997, 3357), ('0_4', 0, 14848), ('2997_6', 2997, 5469), ('13548_9', 13548, 14848), ('6929_2', 6929, 10455), ('4700_7', 4700, 6607), ('13207_11', 13207, 14848), ('6903_3', 6903, 11643), ('13952_5', 13952, 14848), ('6040_9', 6040, 7866), ('7897_7', 7897, 14848), ('5872_2', 5872, 7867), ('12776_6', 12776, 14790), ('865_7', 865, 2978), ('7755_5', 7755, 12728), ('6929_1', 6929, 8111), ('3328_12', 3328, 7688), ('0_7', 0, 3119), ('2540_6', 2540, 2892)]
0
500
1000
1500
2000
2500
3000
3500
4000
4500


/Users/friederikebuck/Desktop/Thesis/FERAL_scripts/Final/scripts/VectorandImageHandlers/NPCurveHandler.py:116: RuntimeWarning: invalid value encountered in divide
  vecs = vecs/norm_cat


0
500
1000
1500
2000


/Users/friederikebuck/miniconda3/envs/worm_tracker_env/lib/python3.8/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/friederikebuck/miniconda3/envs/worm_tracker_env/lib/python3.8/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


0
500
1000
0
500
1000
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
int(self.invert_bool) != int(invert_bool_2 )
self.chunk_start_i 4257
self.chunk_end_i 5063
0
500
1000
1500
2000
2500
3000
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
int(self.invert_bool) != int(invert_bool_2 )
self.chunk_start_i 2375
self.chunk_end_i 3694
int(self.invert_bool) != int(invert_bool_2 )
self.chunk_start_i 3697
self.chunk_end_i 6797
int(self.invert_bool) != int(invert_bool_2 )
self.chunk_start_i 6909
self.chunk_end_i 7020
0
500
1000
1500
2000
2500
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
8500
9000
9500
10000
10500
11000
11500
12000
12500
int(self.invert_bool) != int(invert_bool_2 )
self.chunk_start_i 1221
self.chunk_end_i 1789
int(self.invert_bool) != int(invert_bool_2 )
self.chunk_start_i 3692
self.chunk_end_i 3961
int(self.invert_bool) != int(invert_bool

In [ ]:

def save_as_csv(array, filename):
    reshaped_array = array.reshape(array.shape[0], -1)  # (n_frames, n_pts * 2)
    
    # Save the reshaped array as a CSV file
    np.savetxt(filename, reshaped_array, delimiter=",")

csv_save_dir = os.path.join(parent_dir, "features")#params
os.makedirs(csv_save_dir, exist_ok=True)

for feature_label, mat in [("centroids", centroids_mat), 
                      ("head_coord", head_coord_mat), 
                      ("tail_coord", tail_coord_mat), 
                     ( "mdpt_coord" , mdpt_coord_mat), 
                      ("is_looping", is_looping_mat)
                
                      
                      ]:
    
    feature_csv =  os.path.join(csv_save_dir,f"{feature_label}.csv")
    save_as_csv(mat, feature_csv)